# C8-embeddings — Practice p08 — Solution

In [ ]:
import os, pathlib
_env_root = os.environ.get("USAAIO_BOOK_ROOT")
if _env_root:
    _root = pathlib.Path(_env_root).resolve()
else:
    _start = pathlib.Path.cwd().resolve()
    _root = next(
        p for p in [_start, *_start.parents]
        if (p / "syllabus.md").is_file() and (p / "curriculum").is_dir()
    )
os.environ["GENSIM_DATA_DIR"] = str(_root / "reference" / "cache" / "gensim")

import numpy as np
import gensim.downloader

In [ ]:
kv = gensim.downloader.load("glove-wiki-gigaword-100")

WORDS = ["thunder", "breeze", "drizzle", "hail", "fog", "frost"]
V = np.asarray(kv[WORDS], dtype=np.float64)
row_norms = np.sqrt((V * V).sum(axis=1, keepdims=True))
W = V / row_norms
norms_flat = np.sqrt((V * V).sum(axis=1))
longest = WORDS[int(np.argmax(norms_flat))]
shortest = WORDS[int(np.argmin(norms_flat))]
centroid = W.mean(axis=0)
centroid_norm = float(np.sqrt((centroid * centroid).sum()))
centroid_unit = centroid / centroid_norm
closest = WORDS[int(np.argmax(W @ centroid_unit))]

The unit rows partially cancel when averaged, leaving a centroid norm near 0.703. Comparing rows to the normalized centroid identifies `fog` as most aligned with the group's central direction.

### Answer check

In [ ]:
assert V.shape == W.shape == (6, 100)
assert V.dtype == W.dtype == np.float64
assert norms_flat.shape == (6,)
assert (longest, shortest, closest) == ("drizzle", "frost", "fog")
assert centroid.shape == (100,)
assert np.isclose(centroid_norm, 0.702980878956168, atol=1e-12, rtol=0)
assert centroid_norm < 1.0